# **Introducción a Keras**

En este cuaderno vamos a introducir conceptos básicos para creación, entrenamiento y evaluación de modelos. Para ello, haremos uso del conjunto de datos de la Flor de Iris ya conocido.

# **El conjunto de datos**

El conjunto de datos podemos obtenerlo directamente desde SKLearn. Ahí podemos hacer todo el preprocesamiento necesario y después transformar los datos a tensores.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

dataset= load_iris()
X= dataset['data']
y= dataset['target']
data= zip(X, y)

Xtrain, Xtest, Ytrain, Ytest= train_test_split(X, y, test_size= 0.2, stratify=y)
print('Muestras de training: ', Xtrain.shape, Ytrain.shape)
print('Muestras de test: ', Xtest.shape, Ytest.shape)





Muestras de training:  (120, 4) (120,)
Muestras de test:  (30, 4) (30,)


Si queremos usar conjunto de validación al entrenar con Tensorflow, el parámetro **validation_split** de la función **fit** escoge **LOS ÚLTIMOS DATOS DEL CONJUNTO DE TRAINING**.

Esto puede ser arriesgado si en dicho conjunto faltan patrones de algunas clases o si, en general, la muestra del conjunto de entrenamiento no está uniformemente distribuida (suele pasar con conjuntos de datos pequeños con frecuencia).

Lo vamos a solucionar volviendo a dividir el conjunto de entrenamiento en *training* y *validation* con estratificación, y concatenando ambos.

In [ ]:
import numpy as np

Xt, Xv, Yt, Yv= train_test_split(Xtrain, Ytrain, test_size=0.2, stratify=Ytrain)

Xtrain= np.vstack((Xt, Xv))
Ytrain= np.concatenate((Yt, Yv))
print('Muestras de training: ', Xtrain.shape, Ytrain.shape)


Muestras de training:  (120, 4) (120,)


Necesitamos pasar las salidas a codificación OneHot, dado que la red que vamos a diseñar tendrá como salida una terna indicando la probabilidad de que el patrón de entrada sea de un tipo de flor u otro.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe= OneHotEncoder(sparse_output=False)
ohe.fit(Ytrain.reshape(-1, 1))
Ytrain= ohe.transform(Ytrain.reshape(-1, 1))
Ytest= ohe.transform(Ytest.reshape(-1, 1))

print('Forma de las salidas: training {}  y test {}'.format(Ytrain.shape, Ytest.shape))


Forma de las salidas: training (120, 3)  y test (30, 3)


Por último, lo pasamos a tensor


In [ ]:
import tensorflow as tf

tXtrain= tf.convert_to_tensor(Xtrain)
tYtrain= tf.convert_to_tensor( Ytrain, dtype=tf.int64 )
tXtest= tf.convert_to_tensor(Xtest)
tYtest= tf.convert_to_tensor( Ytest, dtype=tf.int64 )

print('Muestras de training: ', tXtrain.shape, tYtrain.shape)
print('Muestras de test: ', tXtest.shape, tYtest.shape)

Muestras de training:  (120, 4) (120, 3)
Muestras de test:  (30, 4) (30, 3)


# **Creación del modelo**

Crearemos una red neuronal feedforward con 2 capas: oculta y de salida.

- La capa oculta estará completamente conectada (capa densa) y tendrá 50 neuronas activadas por ReLU
- La capa de salida completamente conectada (capa densa) y tendrá 3 neuronas activadas por SoftMax.
  - La primera neurona devolverá la probabilidad de que el especimen de entrada sea *Setosa* (y=0)
  - La segunda neurona devolverá la probabilidad de que el especimen de entrada sea *Versicolor* (y=1)
  - La tercera neurona devolverá la probabilidad de que el especimen de entrada sea *Virginica* (y=2)

In [ ]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense

model= Sequential()
model.add( Input(shape=(Xtrain.shape[1],)))
model.add( Dense(50, activation='relu') )
model.add( Dense(3, activation='softmax') )


# **Selección de función de pérdida**

La función de pérdida a usar será la entropía cruzada categórica, dado que tenemos un problema de clasificación multiclase. Su fórmula es la siguiente:

$CCE= -\sum_{i=1}^C y_i p_i$

Donde $C$ es el número de clases, $y_i=1$ si el patrón es de la clase $i$, y $p_i$ es la probabilidad dada por el modelo para predecir la clase $i$ dado un patrón de entrada.

La documentación de la función de pérdida ```categorical_cross_entropy``` de Keras se encuentra en <a href="https://www.tensorflow.org/api_docs/python/tf/keras/losses/categorical_crossentropy">https://www.tensorflow.org/api_docs/python/tf/keras/losses/categorical_crossentropy</a>

In [ ]:
from tensorflow.keras.losses import categorical_crossentropy

loss_function= categorical_crossentropy

# **Algoritmo de aprendizaje**

Usaremos el descenso de gradiente estocástico (SGD). Estudiad su funcionamiento y la diferencia con el descenso de gradiente clásico en (por ejemplo) el siguiente enlace:

<a href="https://mohitmishra786687.medium.com/stochastic-gradient-descent-a-basic-explanation-cbddc63f08e0">https://mohitmishra786687.medium.com/stochastic-gradient-descent-a-basic-explanation-cbddc63f08e0</a>

La documentación del procedimiento SGD se encuentra en el siguiente enlace:

<a href="https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD">https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD</a>

Usaremos una tasa de aprendizaje estándar de $\lambda=0.1$


In [ ]:
from tensorflow.keras.optimizers import SGD

# Tasa de aprendizaje
lmbda= 0.1
optimizer= SGD(learning_rate= lmbda)


# **Selección de métricas para evaluación**


Por último, seleccionaremos algunas métricas de interés para evaluar el modelo. En particular:


- Accuracy
- Precision
- Recall
- F1Score

Más información sobre las métricas en <a href="https://www.tensorflow.org/api_docs/python/tf/keras/metrics">https://www.tensorflow.org/api_docs/python/tf/keras/metrics</a> .

In [ ]:
metrics= ['accuracy', 'precision', 'recall', 'f1_score']

# **Compilación del modelo y revisión**

Con todos los ingredientes seleccionados, ahora podemos compilar el modelo y visualizar un resumen del mismo.

In [ ]:


model.compile(loss= loss_function, optimizer= optimizer, metrics= metrics)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 50)                  │             250 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             153 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 403 (1.57 KB)

 Trainable params: 403 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

# **Entrenamiento**

Realizaremos un entrenamiento en 100 épocas, diviendo el conjunto en batches de 20 en 20 y utilizando el 20% de los datos para validación.

In [ ]:

bs= 20 # Tamaño del batch
epochs= 100 # Épocas de entrenamiento
val_rate= 0.2 # Tasa de patrones para validación

model.fit(Xtrain, Ytrain, batch_size= bs, epochs=epochs, validation_split= val_rate)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - accuracy: 0.4424 - f1_score: 0.4074 - loss: 1.7281 - precision: 0.4054 - recall: 0.3660 - val_accuracy: 0.6667 - val_f1_score: 0.5333 - val_loss: 0.7432 - val_precision: 1.0000 - val_recall: 0.3750
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7778 - f1_score: 0.7624 - loss: 0.7272 - precision: 0.9737 - recall: 0.3556 - val_accuracy: 0.6667 - val_f1_score: 0.5556 - val_loss: 0.6548 - val_precision: 0.6667 - val_recall: 0.6667
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7299 - f1_score: 0.6920 - loss: 0.5995 - precision: 0.7394 - recall: 0.5847 - val_accuracy: 1.0000 - val_f1_score: 1.0000 - val_loss: 0.5011 - val_precision: 1.0000 - val_recall: 0.9583
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8160 - f1_score: 0.7931 - loss: 0.4859 - precision: 0.8602 - recall: 0.8035 - val_accuracy: 0.7083 - val_f1_score: 0.6393 - val_loss: 0.4833 - val_precision: 0.6957 - val_recall: 0.6667

# **Evaluación**

Vemos cómo se comporta el algoritmo con las métricas seleccionadas.

In [ ]:
ScoresTrain= model.evaluate(Xtrain, Ytrain)
ScoresTest= model.evaluate(Xtest, Ytest)
print('CONJUNTO DE ENTRENAMIENTO')
print('\tLoss Value: {}'.format(ScoresTrain[0]))
for i in range(len(metrics)):
  print('\t{}: {}'.format(metrics[i], ScoresTrain[i+1]))

print('\nCONJUNTO DE TEST')
print('\tLoss Value: {}'.format(ScoresTest[0]))
for i in range(len(metrics)):
  print('\t{}: {}'.format(metrics[i], ScoresTest[i+1]))


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9798 - f1_score: 0.9801 - loss: 0.1020 - precision: 0.9798 - recall: 0.9798 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - f1_score: 1.0000 - loss: 0.0855 - precision: 1.0000 - recall: 1.0000
CONJUNTO DE ENTRENAMIENTO
	Loss Value: 0.09479983896017075
	accuracy: 0.9833333492279053
	precision: 0.9833333492279053
	recall: 0.9833333492279053
	f1_score: [1.         0.97435886 0.9756097 ]

CONJUNTO DE TEST
	Loss Value: 0.08551283925771713
	accuracy: 1.0
	precision: 1.0
	recall: 1.0
	f1_score: [1. 1. 1.]
